In [0]:
from pathlib import Path

# Procura a pasta datasource a partir do diretório atual do notebook.
pastas = [Path.cwd(), *Path.cwd().parents]
candidatos = [p / "datasource" for p in pastas]

pasta_dados = next(
    (p for p in candidatos if any(p.glob("*.csv"))),
    None
)
if pasta_dados is None:
    raise FileNotFoundError(
        f"Não encontrei nenhum arquivo .csv em datasource/ a partir de {Path.cwd()}"
    )

# Encontra todos os arquivos .csv na pasta e usa o nome (sem extensão) como chave
arquivos = {f.stem: f.name for f in sorted(pasta_dados.glob("*.csv"))}

print(f"Arquivos CSV encontrados em: {pasta_dados}:")
print(f"{list(arquivos.values())}")

for nome, arquivo in arquivos.items():
    caminho = pasta_dados / arquivo

    if not caminho.is_file():
        raise FileNotFoundError(f"CSV não encontrado: {caminho}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", "UTF-8")
        .csv(f"file:{caminho}")
    )

    print(f"\n--- {nome} ---")
    print(f"Arquivo: {caminho}")
    print(f"Linhas: {df.count()}")
    print(f"Colunas: {df.columns}")
#    display(df.limit(2))

In [0]:
from pathlib import Path
from pyspark.sql.functions import current_timestamp, lit

catalogo = "will"
esquema = "i30"

tabelas_bronze = {
    "viagens": "bronze_viagens",
    "caronas": "bronze_caronas",
    "abastecimentos": "bronze_abastecimento",
}

for tabela in tabelas_bronze.values():
    nome_completo = f"{catalogo}.{esquema}.{tabela}"
    tabela_sql = f"`{catalogo}`.`{esquema}`.`{tabela}`"

    print(f"\nTabela: {nome_completo} ---")
    display(spark.table(nome_completo).limit(5))

    # print(f"\n--- Descrição e comentários: {nome_completo} ---")
    # display(spark.sql(f"DESCRIBE TABLE EXTENDED {tabela_sql}"))